In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

# Iterate over all files in raw
BASE_DIR = Path("..")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

LEAGUES = ["LEC", "LCK", "LPL", "LCS", "PCS"]

# Sort by year
paths = sorted(RAW_DIR.glob("*_LoL_esports_match_data_from_OraclesElixir.csv"))
print("Found files:")
for p in paths:
    print(" ", p.name)

dfs = []

for p in paths:
    # Try to extract year from filename (adjust if your naming differs)
    m = re.search(r"(\d{4})_", p.name)
    year = int(m.group(1)) if m else None

    tmp = pd.read_csv(p, low_memory=False)
    tmp["year"] = year
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)

# Filter Leagues
df = df[df["league"].isin(LEAGUES)]

# Remove team summary rows (100, 200)
df = df[df["participantid"] <= 10]

# Sanity check
print(f"Total rows: {len(df)}")
print(f"\nMatches per league:")
print(df.groupby("league")["gameid"].nunique())
print(f"\nTotal unique matches: {df['gameid'].nunique()}")
print(f"\nRows per match (should be 10):")
print(df.groupby("gameid").size().value_counts())

# If gameid can repeat across years/files, create a unique key.
df["game_key"] = df["year"].astype(str) + "_" + df["gameid"].astype(str)

print(f"\nTotal unique matches (game_key): {df['game_key'].nunique()}")
print(f"\nRows per match (should be 10):")
print(df.groupby("game_key").size().value_counts().head(10))

Found files:
  2023_LoL_esports_match_data_from_OraclesElixir.csv
  2024_LoL_esports_match_data_from_OraclesElixir.csv
  2025_LoL_esports_match_data_from_OraclesElixir.csv
Total rows: 60390

Matches per league:
league
LCK    1525
LCS     456
LEC     888
LPL    2277
PCS     893
Name: gameid, dtype: int64

Total unique matches: 6039

Rows per match (should be 10):
10    6039
Name: count, dtype: int64

Total unique matches (game_key): 6039

Rows per match (should be 10):
10    6039
Name: count, dtype: int64


C:\Users\QuijadaNelson\AppData\Local\Temp\ipykernel_21240\3469024322.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["game_key"] = df["year"].astype(str) + "_" + df["gameid"].astype(str)


In [3]:
df = df[[
    "gameid", "game_key", "year", "league", "participantid", "side",
    "position", "playerid", "playername", "champion", "result"
]].copy()

needed = ["playerid", "champion", "result", "side", "league", "game_key"]
df = df.dropna(subset=needed)

In [4]:
from sklearn.preprocessing import LabelEncoder

# Map from elixir ids to numeric ids
player_encoder = LabelEncoder()
champ_encoder  = LabelEncoder()
league_encoder = LabelEncoder()

player_encoder.fit(df["playerid"])
champ_encoder.fit(df["champion"])
league_encoder.fit(df["league"])

# Transform strings to integers
df["player_id"] = player_encoder.transform(df["playerid"])
df["champ_id"]  = champ_encoder.transform(df["champion"])
df["league_id"] = league_encoder.transform(df["league"])

# Lookup tables for reversing id to name
playerid_to_name = df.drop_duplicates("playerid").set_index("playerid")["playername"]

player_lookup = pd.DataFrame({
    "player_id":   np.arange(len(player_encoder.classes_)),
    "playerid":    player_encoder.classes_,
    "player_name": [playerid_to_name[pid] for pid in player_encoder.classes_]
})

champ_lookup = pd.DataFrame({
    "champ_id":   np.arange(len(champ_encoder.classes_)),
    "champ_name": champ_encoder.classes_
})

league_lookup = pd.DataFrame({
    "league_id":   np.arange(len(league_encoder.classes_)),
    "league_name": league_encoder.classes_
})

# Save the lookups
player_lookup.to_csv(PROCESSED_DIR / "player_lookup.csv", index=False)
champ_lookup.to_csv(PROCESSED_DIR / "champ_lookup.csv", index=False)
league_lookup.to_csv(PROCESSED_DIR / "league_lookup.csv", index=False)

# Print basic stats
N_players = len(player_encoder.classes_)
N_champs  = len(champ_encoder.classes_)
N_leagues = len(league_encoder.classes_)

print(f"Players: {N_players}")
print(f"Champions: {N_champs}")
print(f"Leagues: {N_leagues} — {list(league_encoder.classes_)}")
print(f"\nSample player mapping:")
print(player_lookup.head(10))

Players: 629
Champions: 165
Leagues: 5 — ['LCK', 'LCS', 'LEC', 'LPL', 'PCS']

Sample player mapping:
   player_id                                   playerid player_name
0          0  oe:player:0015d99e65183977a9e65547b37f1cb        Harp
1          1  oe:player:007be3acd5669d100711b13d79e2336      Sniper
2          2  oe:player:00e1677908b0a579a5870fbad836dcc      QiuQiu
3          3  oe:player:0103d21d116b61eb55fa351242208cf     Kaiwing
4          4  oe:player:010c1b0d37c2b7459b215dae7ffeeac   Cabochard
5          5  oe:player:01e34275fcdf9ccb290e4ede897697a   xiaoyueji
6          6  oe:player:027a5217b659e8bedb70978dea494d6  Irrelevant
7          7  oe:player:030a726de315d9ad86989bf360809e9      Prince
8          8  oe:player:035b9ae578574cdf728d8760483d804      Effort
9          9  oe:player:03b50ebd9393dbcdcd26fe529b40803       Xiamu


In [5]:
# Create lists to save
blue_players_list = [] # Who played on blue side
red_players_list  = [] # Who played on red side
blue_champs_list  = []
red_champs_list   = []
league_ids_list   = [] # The leagues
y_list            = [] # Outcomes (who won)
skipped = 0
skip_reasons = {"wrong_count": 0, "bad_sides": 0}

for game_key, group in df.groupby("game_key"):
    #Skip if a game doesnt have the correct number of entires
    if len(group) != 10:
        skip_reasons["wrong_count"] += 1
        skipped += 1
        continue
    # Sort players in order and which side they played in
    blue = group[group["side"] == "Blue"].sort_values("participantid")
    red  = group[group["side"] == "Red"].sort_values("participantid")

    # Skip bad data
    if len(blue) != 5 or len(red) != 5:
        skip_reasons["bad_sides"] += 1
        skipped += 1
        continue
    
    # Save the data we gathered
    blue_players_list.append(blue["player_id"].values)
    red_players_list.append(red["player_id"].values)
    blue_champs_list.append(blue["champ_id"].values)
    red_champs_list.append(red["champ_id"].values)
    league_ids_list.append(blue["league_id"].iloc[0])
    y_list.append(int(blue["result"].iloc[0]))

# Convert to arrays
blue_players = np.array(blue_players_list)
red_players  = np.array(red_players_list)
blue_champs  = np.array(blue_champs_list)
red_champs   = np.array(red_champs_list)
league_ids   = np.array(league_ids_list)
y            = np.array(y_list)

print(f"Skipped {skipped} games:")
for reason, count in skip_reasons.items():
    print(f"  {reason}: {count}")

print(f"\nFinal dataset: {len(y)} matches")
print(f"Blue winrate: {y.mean():.3f}")
print(f"\nMatches per league:")
for lid in range(N_leagues):
    lname = league_encoder.classes_[lid]
    count = (league_ids == lid).sum()
    print(f"  {lname}: {count}")

Skipped 10 games:
  wrong_count: 10
  bad_sides: 0

Final dataset: 6028 matches
Blue winrate: 0.530

Matches per league:
  LCK: 1524
  LCS: 456
  LEC: 888
  LPL: 2277
  PCS: 883


In [6]:
# Save as npy files
np.save(PROCESSED_DIR / "blue_players.npy", blue_players)
np.save(PROCESSED_DIR / "red_players.npy",  red_players)
np.save(PROCESSED_DIR / "blue_champs.npy",  blue_champs)
np.save(PROCESSED_DIR / "red_champs.npy",   red_champs)
np.save(PROCESSED_DIR / "league_ids.npy",   league_ids)
np.save(PROCESSED_DIR / "y.npy",            y)

print("Saved to:", PROCESSED_DIR)
print("\nFiles:")
for f in sorted(PROCESSED_DIR.glob("*")):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

print(f"\nDataset summary:")
print(f"  Matches:   {len(y)}")
print(f"  Players:   {N_players}")
print(f"  Champions: {N_champs}")
print(f"  Leagues:   {N_leagues}")

Saved to: ..\data\processed

Files:
  blue_champs.npy (235.6 KB)
  blue_players.npy (235.6 KB)
  champ_lookup.csv (1.9 KB)
  league_ids.npy (47.2 KB)
  league_lookup.csv (0.1 KB)
  player_lookup.csv (32.8 KB)
  red_champs.npy (235.6 KB)
  red_players.npy (235.6 KB)
  y.npy (47.2 KB)

Dataset summary:
  Matches:   6028
  Players:   629
  Champions: 165
  Leagues:   5
